<a href="https://colab.research.google.com/github/sn-kc/AI-ML-CLASS/blob/main/MLP_Boston_data_(Keras_API).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step-1: Loading libraries

In [1]:
import  numpy  as  np
import  pandas  as  pd
import  matplotlib.pyplot  as  plt
import  tensorflow  as  tf
from  tensorflow  import  keras

## Step-2 Loading dataset

In [2]:
# Load the dataset from url
# Target URL for the original dataset
data_url = "http://lib.stat.cmu.edu/datasets/boston"
raw_df = pd.read_csv(data_url, sep=r"\s+", skiprows=22, header=None)

# Process features (X) and target variable (y) due to the dataset's formatting
X = np.hstack([raw_df.values[::2, :], raw_df.values[1::2, :2]])
y = raw_df.values[1::2, 2]

print("Features shape:", X.shape)  # Output: (506, 13)
print("Target shape:", y.shape)    # Output: (506,)

Features shape: (506, 13)
Target shape: (506,)


# Step-3 Data Preprocessing

In [3]:
# Setting the random seeds for repeatability
tf.random.set_seed(42)
np.random.seed( 42 )

# Split the data into 80% training and 20% testing
from  sklearn.model_selection  import  train_test_split

X_train, X_test, y_train, y_test  =  train_test_split(X, y, test_size = 0.2 ,
                                                     random_state = 42 )

# Step-4  Feature Scaling- for better results on Neural Network

In [4]:
# Feature scaling using Standardization
from  sklearn.preprocessing  import  StandardScaler
sc  =  StandardScaler()

# Training the feature scaling parameters
sc.fit(X_train)

# Applying transformations to both training and testing set
X_train_std  =  sc.transform(X_train)

X_test_std  =  sc.transform(X_test)


Step 5 Neural Net Architecture

In [5]:
X_train.shape[1:]

(13,)

# Neural network layer adjustment

Small datasets are highly prone to overfitting, where the model memorizes the training data but performs poorly on the test data. Adding a Dropout layer randomly turns off neurons during training, forcing the network to learn more robust patterns.

Slightly Increased Network Capacity

In [21]:
# Create neural network using keras API
# Sequential() does linear stacking of layers
model_MLP  =  keras.models.Sequential()
# Hidden layer definitions
# Hidden Layer 1
model_MLP.add(keras.layers.Dense(units=64, activation='relu', input_shape=X_train.shape[1:]))
model_MLP.add(keras.layers.Dropout(0.2))  # Prevents overfitting
# Hidden Layer 2
model_MLP.add(keras.layers.Dense(units=32, activation='relu'))
model_MLP.add(keras.layers.Dropout(0.2))  # Prevents overfitting

#model_MLP.add(keras.layers.Dense(units = 25 , activation = 'relu' ,
                                 #input_shape =  X_train.shape[ 1 :]))

#model_MLP.add(keras.layers.Dense(units = 5 , activation = 'relu' , ))

# Output layer definitions
model_MLP.add(keras.layers.Dense(units = 1 , activation = 'linear' ))# regression model is required


# Print the summary of network architecture
model_MLP.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 64)             │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,009 (11.75 KB)

 Trainable params: 3,009 (11.75 KB)

 Non-trainable params: 0 (0.00 B)

# Step-6 Model Compiling Training

Increase epochs . This gives the model more time to minimize the error.
Adjusting the Learning Rate for minimum error

In [31]:
# Compile the network model with relevant configurations.
# loss, optimizer and metrics are three important configurations.

#model_MLP.compile(loss = 'mse' , optimizer = 'adam' ,
                  #metrics = [ 'mae' ])# mae- mean absolute error, mse- mean square error
model_MLP.compile(
    loss='mse',
    optimizer=keras.optimizers.Adam(learning_rate=0.005),
    metrics=['mae']
)

model_MLP.fit(x = X_train_std, y = y_train, validation_split = 0.1 ,
              epochs = 200 , batch_size = 16,verbose=False )

#Step-7  Model Evaluation

In [33]:
test_loss, test_error  =  model_MLP.evaluate(x = X_test_std, y = y_test)

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 11.8269 - mae: 2.3599


#Step-8 Model Accuracy and loss

In [34]:
# The accuracy obtained can be close enough to what is obtained here.
print (test_loss, test_error)

11.826885223388672 2.3599112033843994


In [35]:
modelout=model_MLP.predict(X_test_std)

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step


In [37]:
print(modelout)

[[25.295385 ]
 [33.033203 ]
 [16.145235 ]
 [21.727123 ]
 [17.221424 ]
 [18.122904 ]
 [17.369549 ]
 [17.614721 ]
 [24.447926 ]
 [18.621113 ]
 [21.408096 ]
 [18.964836 ]
 [ 7.6468244]
 [18.136116 ]
 [17.722235 ]
 [21.890112 ]
 [19.132826 ]
 [11.685037 ]
 [42.121193 ]
 [12.617523 ]
 [24.09376  ]
 [24.31025  ]
 [13.381839 ]
 [22.540333 ]
 [15.558323 ]
 [17.397972 ]
 [18.445183 ]
 [12.209776 ]
 [20.73935  ]
 [17.69648  ]
 [23.76032  ]
 [21.504004 ]
 [19.606756 ]
 [23.62139  ]
 [16.542227 ]
 [16.528269 ]
 [29.098467 ]
 [20.115505 ]
 [19.975193 ]
 [22.855474 ]
 [16.707373 ]
 [28.170435 ]
 [46.744183 ]
 [17.723392 ]
 [24.458277 ]
 [12.673061 ]
 [14.424547 ]
 [24.219412 ]
 [18.835796 ]
 [22.369476 ]
 [20.231518 ]
 [31.652826 ]
 [16.158813 ]
 [23.259726 ]
 [41.912945 ]
 [20.907904 ]
 [16.078928 ]
 [30.133905 ]
 [22.963585 ]
 [16.243992 ]
 [24.119265 ]
 [29.97328  ]
 [30.227945 ]
 [15.661823 ]
 [22.130234 ]
 [19.44995  ]
 [13.006359 ]
 [21.77161  ]
 [26.334597 ]
 [14.02673  ]
 [21.616293 ]
 [28.7

In [12]:
print(y_test)

[23.6 32.4 13.6 22.8 16.1 20.  17.8 14.  19.6 16.8 21.5 18.9  7.  21.2
 18.5 29.8 18.8 10.2 50.  14.1 25.2 29.1 12.7 22.4 14.2 13.8 20.3 14.9
 21.7 18.3 23.1 23.8 15.  20.8 19.1 19.4 34.7 19.5 24.4 23.4 19.7 28.2
 50.  17.4 22.6 15.1 13.1 24.2 19.9 24.  18.9 35.4 15.2 26.5 43.5 21.2
 18.4 28.5 23.9 18.5 25.  35.4 31.5 20.2 24.1 20.  13.1 24.8 30.8 12.7
 20.  23.7 10.8 20.6 20.8  5.  20.1 48.5 10.9  7.  20.9 17.2 20.9  9.7
 19.4 29.  16.4 25.  25.  17.1 23.2 10.4 19.6 17.2 27.5 23.  50.  17.9
  9.6 17.2 22.5 21.4]
